# YOLOv8 Experiments: GOST Stamp Detection

**Цель:** Обучить YOLOv8 для детекции штампов на строительных чертежах.

**v2 (исторический):** 500 synthetic (80/20 train/val) + 49 real (test, 4 donors excluded)
**v3 (чистый baseline):** 500 synthetic (all train) + 10 real stratified val (val_honest) + 35 real test (4 donors + 10 val excluded)

**Метрики:** IoU, Precision, Recall, F1 на 35 реальных non-donor, non-val изображениях

**Подход:** Single training run, yolov8n, 50 epochs, GPU T4 (Colab)


# 1. Setup

⚠️ **Запустить только один раз!** Клонирует репозиторий (sparse checkout) и устанавливает зависимости.


In [1]:
import sys
import random
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    %cd /content
    !rm -rf aie-group-2-sapar
    !git init aie-group-2-sapar
    %cd aie-group-2-sapar
    !git sparse-checkout set project
    !git remote add origin https://github.com/Sapar-hub/aie-group-2-sapar.git
    !git pull origin main
    %cd project
    !pip install -q ultralytics opencv-python-headless pyyaml
    PROJECT_DIR = Path.cwd()
    sys.path.insert(0, str(PROJECT_DIR / "src"))
else:
    PROJECT_DIR = Path.cwd().parent
    sys.path.insert(0, str(PROJECT_DIR / "src"))


In [2]:
# Load pre-generated training data from Google Drive (Colab only)
if IN_COLAB and not (Path('data') / 'images' / 'train_v3').exists():
    from google.colab import drive
    drive.mount('/content/drive')
    import shutil
    DRIVE_DATA = Path('/content/drive/MyDrive/aie-group-2-data')
    if (DRIVE_DATA / 'images' / 'train_v3').exists():
        for subdir in ['images', 'labels']:
            shutil.copytree(str(DRIVE_DATA / subdir), str(Path('data') / subdir), dirs_exist_ok=True)
        print("Training data loaded from Google Drive")
    else:
        print("=" * 60)
        print("No pre-generated data found on Drive!")
        print("Run notebooks/exp02_synthetic_data.ipynb first to generate it.")
        print("=" * 60)


## 2. Imports & Data Setup

In [3]:
import numpy as np
import matplotlib.pyplot as plt
import yaml
from ultralytics import YOLO

from evaluation.metrics import DetectionResult, bbox_iou, yolo_to_pixel, compute_metrics, print_metrics
from data.loader import load_image_and_labels

RANDOM_STATE = 42

with open(PROJECT_DIR / "configs" / "config.yaml") as f:
    CFG = yaml.safe_load(f)

DATA_DIR = PROJECT_DIR / "data"
ARTIFACTS_DIR = PROJECT_DIR / "artifacts"
ARTIFACTS_DIR.mkdir(exist_ok=True)
(ARTIFACTS_DIR / "models").mkdir(exist_ok=True)
(ARTIFACTS_DIR / "metrics").mkdir(exist_ok=True)
(ARTIFACTS_DIR / "figures").mkdir(exist_ok=True)

IMAGE_TEST_DIR = DATA_DIR / "images" / "test"
LABEL_TEST_DIR = DATA_DIR / "labels" / "test"

print(f"Working dir: {PROJECT_DIR}")
print(f"Test images: {len(list(IMAGE_TEST_DIR.glob('*.png'))) + len(list(IMAGE_TEST_DIR.glob('*.jpg')))}")
print(f"Test labels: {len(list(LABEL_TEST_DIR.glob('*.txt')))}")

Working dir: /home/saparch/playground/aie-group-2-sapar/project
Test images: 49
Test labels: 49


## 3. Data, Donors & Split (v3 Clean Baseline)

**Donors (4):** stamp sources for copy-paste synthesis, excluded from eval.
**Val (10):** stratified-selected real images for YOLO validation during training.
**Test (35):** remaining non-donor, non-val images for final evaluation.

All 49 real images loaded for inference; 14 excluded from metrics (4 donors + 10 val) → **eval on 35 test images**.


In [4]:
# Donors and val images are excluded from eval metrics
# Stratified val selection runs in exp02_synthetic_data.ipynb
DONORS = {"test_11.png", "test_17.png", "test_23.png", "test_42.png"}
print(f"Donors (stamp sources, excluded from eval): {sorted(DONORS)}")

# Val images are selected by select_donors() with exclude=DONORS, n_donors=10, seed=42
# They are copied to data/images/val_honest/ by exp02
VAL_HONEST_DIR = DATA_DIR / "images" / "val_honest"
VAL_IMAGES = set(p.name for p in sorted(VAL_HONEST_DIR.glob("*.png")) + sorted(VAL_HONEST_DIR.glob("*.jpg")))
print(f"Val images (real, stratified, excluded from test eval): {sorted(VAL_IMAGES)}")
print(f"Test set: 49 - {len(DONORS)} - {len(VAL_IMAGES)} = {49 - len(DONORS) - len(VAL_IMAGES)} images")


Donors (stamp sources, excluded from eval): ['test_09.jpg', 'test_14.png', 'test_34.jpg', 'test_49.jpg']
Val images (real, stratified, excluded from test eval): ['test_04.png', 'test_07.png', 'test_08.png', 'test_11.png', 'test_23.png', 'test_29.jpg', 'test_35.jpg', 'test_38.jpg', 'test_44.png', 'test_46.png']
Test set: 49 - 4 - 10 = 35 images


In [5]:
# All 49 real images loaded for inference; 4 donors + 10 val excluded from metrics
import glob
all_images = sorted(IMAGE_TEST_DIR.glob("*.png")) + sorted(IMAGE_TEST_DIR.glob("*.jpg"))
test_images = all_images  # no holdout split, DONORS + VAL_IMAGES filter in metrics

EXCLUDE = DONORS | VAL_IMAGES
print(f"Test images: {len(test_images)} (eval on {len(test_images) - len(EXCLUDE)} non-donor, non-val after filter)")


Test images: 49 (eval on 35 non-donor, non-val after filter)


## 4. Training (v3)

Запускаем YOLOv8n на данных из `gost_stamp.yaml`.
Параметры: `rect=True` (сохраняет пропорции A4 vs A1), `mosaic=0.0` (не режет мелкие штампы), `seed=42` (воспроизводимость).

**v3 отличие:** val теперь указывает на 10 реальных изображений (val_honest), а не на синтетику.


In [6]:
from models.train_yolo import train_yolo
import time
start = time.time()

best_pt = train_yolo(
    config_path=PROJECT_DIR / "configs" / "config.yaml",
    project=str(ARTIFACTS_DIR / "yolo"),
    name="exp04",
)

elapsed = time.time() - start
print(f"\nTraining time: {elapsed/60:.1f} minutes")

New https://pypi.org/project/ultralytics/8.4.53 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.42 🚀 Python-3.12.12 torch-2.11.0+cu130 CPU (Intel Core i5-9400F 2.90GHz)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/saparch/playground/aie-group-2-sapar/project/data/gost_stamp.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, mome

## 5. Evaluation — Score Threshold Sweep + Greedy Matching

Загружаем лучшие веса, оцениваем на 35 test non-donor (4 донора + 10 val исключены из метрик).
Sweep по `conf ∈ [0.05, 0.1, 0.2, 0.3]`, выбор по F1.
Greedy matching: из нескольких предсказаний выбираем с лучшим IoU к GT.
Fallback: если greedy не нашёл — берём первый prediction (highest confidence).


In [7]:
from evaluation.evaluate_yolo import evaluate_yolo

model = YOLO(str(best_pt))
print(f"Loaded weights from {best_pt}")

best_metrics, results_list = evaluate_yolo(
    model, IMAGE_TEST_DIR, LABEL_TEST_DIR, DONORS
)
print_metrics(best_metrics, prefix="\nYOLO ")
metrics = best_metrics

Loaded weights from /home/saparch/playground/aie-group-2-sapar/project/artifacts/yolo/exp04/weights/best.pt

YOLO Images:         45

YOLO Detected:        35 (77.8%)

YOLO IoU mean:        0.700

YOLO IoU std:         0.376

YOLO IoU median:      0.890

YOLO IoU >= 0.5:      77.8%

YOLO Precision:       1.000

YOLO Recall:          0.778

YOLO F1:              0.875


## 6. IoU Threshold Analysis

In [8]:
print("IoU @ different thresholds:")
for thresh in [0.3, 0.5, 0.75]:
    m = compute_metrics(results_list, iou_threshold=thresh)
    print(f"  IoU >= {thresh}: {m.get('iou_at_threshold', 0)*100:.1f}%")

ious = [r.iou for r in results_list]
print(f"\nIoU stats: mean={np.mean(ious):.3f}, std={np.std(ious):.3f}, median={np.median(ious):.3f}")
print(f"Detection rate: {sum(1 for r in results_list if r.found)}/{len(results_list)}")

IoU @ different thresholds:
  IoU >= 0.3: 77.8%
  IoU >= 0.5: 77.8%
  IoU >= 0.75: 77.8%

IoU stats: mean=0.700, std=0.376, median=0.890
Detection rate: 35/45


## 7. Visualization

In [9]:
import cv2

sorted_results = sorted(results_list, key=lambda r: r.iou)
worst = sorted_results[0]
best = sorted_results[-1]

fig, axes = plt.subplots(1, 2, figsize=(16, 8))

for ax, result, title_prefix in zip(axes, [worst, best], ["Worst", "Best"]):
    img_path = [p for p in test_images if p.name == result.image_name][0]
    img, _ = load_image_and_labels(img_path, LABEL_TEST_DIR)
    vis = img.copy()
    
    if result.gt_bbox:
        x, y, bw, bh = result.gt_bbox
        cv2.rectangle(vis, (x, y), (x+bw, y+bh), (0, 255, 0), 3)
    if result.pred_bbox:
        x, y, bw, bh = result.pred_bbox
        cv2.rectangle(vis, (x, y), (x+bw, y+bh), (0, 0, 255), 2)
    
    ax.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
    ax.set_title(f"{title_prefix} IoU={result.iou:.3f}")
    ax.axis("off")

plt.suptitle("Green=GT, Red=Pred (YOLO)")
plt.tight_layout()
plt.savefig(ARTIFACTS_DIR / "figures" / "yolo_best_worst.png", dpi=150)
plt.show()

<Figure size 1600x800 with 2 Axes>

## 8. Conclusions (v3 Clean Baseline)

Threshold sweep + greedy matching на 35 test non-donor images (4 donors + 10 val excluded).
Лучший `conf` выбран по F1.

**v3 отличие:** val split — 10 реальных изображений (val_honest, стратифицированный отбор).
Тест — 35 реальных изображений, никогда не использовавшихся в обучении.


In [10]:
summary = {
    "model": "YOLOv8n",
    "data": "500 synthetic (all train) + 49 real (eval on 35 test non-donor, non-val)",
    "epochs": 50,
    "imgsz": 640,
    "rect": True,
    "mosaic": 0.0,
    "train_time_min": round(elapsed/60, 1),
    "iou_mean": round(metrics['iou_mean'], 3),
    "iou_std": round(metrics['iou_std'], 3),
    "precision": round(metrics['precision'], 3),
    "recall": round(metrics['recall'], 3),
    "f1": round(metrics['f1'], 3),
    "detection_rate": round(metrics['detection_rate'], 3),
}

import json
with open(ARTIFACTS_DIR / "metrics" / "yolo_v3_results.json", "w") as f:
    json.dump(summary, f, indent=2)

print("Summary saved to artifacts/metrics/yolo_v3_results.json")
print(json.dumps(summary, indent=2))

Summary saved to artifacts/metrics/yolo_v3_results.json
{
  "model": "YOLOv8n",
  "data": "500 synthetic (all train) + 49 real (eval on 35 test non-donor, non-val)",
  "epochs": 50,
  "imgsz": 640,
  "rect": true,
  "mosaic": 0.0,
  "train_time_min": 88.8,
  "iou_mean": 0.7,
  "iou_std": 0.376,
  "precision": 1.0,
  "recall": 0.778,
  "f1": 0.875,
  "detection_rate": 0.778
}
